In [55]:
import requests
import json
import os
from csv import writer

from requests.auth import HTTPBasicAuth
import csv
# Load the Pandas libraries with alias 'pd' 
import pandas as pd

In [56]:
df = pd.read_csv("2025Q1 COP24 Q1 DATIM File October to december 2024.csv")

C:\Users\Shapr0019\AppData\Local\Temp\ipykernel_18020\1099284769.py:1: DtypeWarning: Columns (10,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("2025Q1 COP24 Q1 DATIM File October to december 2024.csv")


In [57]:
df2 = df

In [58]:
df2.head(2)

,Period,Funding_Mechanism,SiteProvince,SiteDistrict,SiteName,Org_Unit,Indicator,DATIM_Indicator,Disaggregation,ElementType,ElementResultGroup,ElementAge,ElementGender,ElementLocation,ElementModality,Value
0,2025Q1,70465,Matabeleland North,Nkayi,Mbuma - 100998 - Mission Hospital,dhwL8YFJUKy,CXCA_SCRN TA,CXCA_SCRN_N_TA_Age_Sex_HIV_Scrn_Visit,dh4TQ68p2SC,TA,Cervical Cancer - Negative,15-19,Female,Facility,First Time Screened,0
1,2025Q1,70465,Matabeleland North,Nkayi,Mbuma - 100998 - Mission Hospital,dhwL8YFJUKy,CXCA_SCRN TA,CXCA_SCRN_N_TA_Age_Sex_HIV_Scrn_Visit,pdCeAB4EYYM,TA,Cervical Cancer - Negative,20-24,Female,Facility,First Time Screened,0


### Remove uncessary columns 

In [59]:
# List the columns you want to drop
columns_to_drop = ['Funding_Mechanism', 'SiteProvince', 'ElementType','ElementLocation','Indicator']  # Replace with your actual column names

# Drop the specified columns
df2 = df2.drop(columns=columns_to_drop)

### COC name

In [61]:
# Ensure age and gender columns are strings
df2['ElementAge'] = df2['ElementAge'].astype(str)
df2['ElementGender'] = df2['ElementGender'].astype(str)

# Create the 'Category Option Combo Name' column by concatenating 'age' and 'gender'
df2['categoryOptionComboEHR'] = df2['ElementAge'] + ', ' + df['ElementGender']

#### COC mmd

In [63]:
# Ensure age and gender columns are strings
df2['ElementAge'] = df2['ElementAge'].astype(str)
df2['ElementGender'] = df2['ElementGender'].astype(str)
df2['ElementModality'] = df2['ElementModality'].astype(str)

# Create the 'Category Option Combo Name' column by concatenating 'age' and 'gender'
df2['categoryOptionComboEHRmmd'] = df2['ElementAge'] + ', ' + df['ElementGender']+ ', ' + df['ElementModality']
#df2['categoryOptionComboEHR'] = df2['ElementAge'] + ', ' + df['ElementGender']

#### COC Tx_new CD4

In [64]:
# Ensure age and gender columns are strings
df2['ElementAge'] = df2['ElementAge'].astype(str)
df2['ElementGender'] = df2['ElementGender'].astype(str)
df2['ElementResultGroup'] = df2['ElementResultGroup'].astype(str)

# Create the 'Category Option Combo Name' column by concatenating 'age' and 'gender'
df2['categoryOptionComboEHRtxnewCD4'] = df2['ElementAge'] + ', ' + df['ElementGender']+ ', ' + df['ElementResultGroup']
#df2['categoryOptionComboEHR'] = df2['ElementAge'] + ', ' + df['ElementGender']

In [66]:
#df2.to_csv('TX_NEW KeyPop66.csv', sep=',',index=False)

In [67]:
#df2 = qw2

## Columns Renaming 

In [68]:
df2.rename(columns={'DATIM_Indicator': 'dataElementEHR'}, inplace=True)
df2.rename(columns={'Org_Unit': 'orgUnitDATIM'}, inplace=True)
df2.rename(columns={'SiteName': 'orgUnit'}, inplace=True)

### Add column DataElement Name in MRF_EHR

In [70]:
# Step 2: Find the position of the column 'orgUnit'
org_unit_index = df2.columns.get_loc('dataElementEHR') + 1  # +1 to insert after 'orgUnit'

# Step 3: Insert the new columns at the desired position
df2.insert(org_unit_index, 'DataElement Name in MRF_EHR', '')  # Insert empty column for 'Org Unit Name'
#df2.insert(org_unit_index + 1, 'Org Unit ID', '')  # Insert empty column for 'Org Unit ID'

In [71]:
# Save the updated DataFrame back to a CSV file if needed
#df2.to_csv('Test.csv', index=False)

## Data Dictionary 
### Mapping dataElemntsEHR code to DHIS2 real name : Vlook, Renaming data element
#### Make sure to update CSV file column named : DataElementCodeEHRDicto

In [73]:
import pandas as pd

# Assuming 'df2' and 'facility_mapping_df' DataFrames are already loaded in your environment
# Load the facility mapping data from the provided CSV file.
facility_mapping_df = pd.read_csv('01mapping EHR_DHIS2 DataDictonery 30_10_2024C1.csv')  # Replace with your file path

# Creating the mapping dictionary from 'facility_mapping_df'
# This time the dictionary maps from DataElementCodeEHRDicto to DataElementNameDict
data_element_code_to_name = dict(zip(facility_mapping_df['01DataElementCodeEHRDicto'], facility_mapping_df['01DataElementNameDict']))

# Iterating over df2 to update 'DataElement Name in MRF_EHR' based on 'dataElementEHR' mapping
for index, row in df2.iterrows():
    # Use the mapping dictionary to find the corresponding name, if available
    data_element_code = row['dataElementEHR']
    if data_element_code in data_element_code_to_name:
        df2.at[index, 'DataElement Name in MRF_EHR'] = data_element_code_to_name[data_element_code]
    # If the data element code is not in the dictionary, leave the value in 'DataElement Name in MRF_EHR' unchanged

# This approach directly assigns the value to 'DataElement Name in MRF_EHR', avoiding the map and potential indexing issues

## Add column Org Unit Name and Org Unit ID

In [80]:
# Step 2: Find the position of the column 'orgUnit'
org_unit_index = df2.columns.get_loc('orgUnit') + 1  # +1 to insert after 'orgUnit'

# Step 3: Insert the new columns at the desired position
df2.insert(org_unit_index, 'Org Unit Name', '')  # Insert empty column for 'Org Unit Name'
df2.insert(org_unit_index + 1, 'Org Unit ID', '')  # Insert empty column for 'Org Unit ID'

In [81]:
df2.head(2)

,Period,SiteDistrict,orgUnit,Org Unit Name,Org Unit ID,orgUnitDATIM,dataElementEHR,DataElement Name in MRF_EHR,Disaggregation,ElementResultGroup,ElementAge,ElementGender,ElementModality,Value,categoryOptionComboEHR,categoryOptionComboEHRmmd,categoryOptionComboEHRtxnewCD4
0,2025Q1,Nkayi,Mbuma - 100998 - Mission Hospital,,,dhwL8YFJUKy,CXCA_SCRN_N_TA_Age_Sex_HIV_Scrn_Visit,"CXCA_SCRN (N, TA, Age/Sex/HIVStatus/ScreenResu...",dh4TQ68p2SC,Cervical Cancer - Negative,15-19,Female,First Time Screened,0,"15-19, Female","15-19, Female, First Time Screened","15-19, Female, Cervical Cancer - Negative"
1,2025Q1,Nkayi,Mbuma - 100998 - Mission Hospital,,,dhwL8YFJUKy,CXCA_SCRN_N_TA_Age_Sex_HIV_Scrn_Visit,"CXCA_SCRN (N, TA, Age/Sex/HIVStatus/ScreenResu...",pdCeAB4EYYM,Cervical Cancer - Negative,20-24,Female,First Time Screened,0,"20-24, Female","20-24, Female, First Time Screened","20-24, Female, Cervical Cancer - Negative"


## Data Dictionary 
### Mapping facility code from EHR to DHIS2 : VlooK 
#### This is fully updated 

In [83]:
import pandas as pd

# Assuming 'df2' is your existing DataFrame that corresponds to the 'Results23' data.

# Load the facility mapping data from the provided CSV file.
#############facility_mapping_df = pd.read_csv('facility_mapping3.csv')  # Replace with your file path

# Clean the 'orgUnit' codes in df2 to avoid mismatches due to case or whitespace issues.
df2['orgUnit'] = df2['orgUnit'].str.strip().str.upper()

# Clean the 'Facility Code' in the facility_mapping_df as well.
facility_mapping_df['Facility Code'] = facility_mapping_df['Facility Code'].str.strip().str.upper()

# Create a dictionary for mapping from facility_mapping_df using 'Facility Code' as keys.
facility_code_to_name = dict(zip(facility_mapping_df['Facility Code'], facility_mapping_df['Org Unit Name']))
facility_code_to_id = dict(zip(facility_mapping_df['Facility Code'], facility_mapping_df['Org Unit ID']))

# Use the map function to create the 'Org Unit Name' and 'Org Unit ID' columns in df2.
df2['Org Unit Name'] = df2['orgUnit'].map(facility_code_to_name)
df2['Org Unit ID'] = df2['orgUnit'].map(facility_code_to_id)

# Now df2 has the 'Org Unit Name' and 'Org Unit ID' fields updated directly, without creating extra columns.


In [84]:
#df2.to_csv('MER data facility444.csv', sep=',',index=False)

## Add columns 

In [86]:
# Step 2: Find the position of the column 'orgUnit'
org_unit_index = df2.columns.get_loc('categoryOptionComboEHR') + 1  # +1 to insert after 'orgUnit'

# Step 3: Insert the new columns at the desired position
df2.insert(org_unit_index, 'categoryOptionComboDHIS2', '')  # Insert empty column for 'Org Unit Name'
#df2.insert(org_unit_index + 1, 'Org Unit ID', '')  # Insert empty column for 'Org Unit ID'

In [87]:
df2.head(2)

,Period,SiteDistrict,orgUnit,Org Unit Name,Org Unit ID,orgUnitDATIM,dataElementEHR,DataElement Name in MRF_EHR,Disaggregation,ElementResultGroup,ElementAge,ElementGender,ElementModality,Value,categoryOptionComboEHR,categoryOptionComboDHIS2,categoryOptionComboEHRmmd,categoryOptionComboEHRtxnewCD4
0,2025Q1,Nkayi,MBUMA - 100998 - MISSION HOSPITAL,mn Mbuma Mission Hospital,MjYcpFjUjbG,dhwL8YFJUKy,CXCA_SCRN_N_TA_Age_Sex_HIV_Scrn_Visit,"CXCA_SCRN (N, TA, Age/Sex/HIVStatus/ScreenResu...",dh4TQ68p2SC,Cervical Cancer - Negative,15-19,Female,First Time Screened,0,"15-19, Female",,"15-19, Female, First Time Screened","15-19, Female, Cervical Cancer - Negative"
1,2025Q1,Nkayi,MBUMA - 100998 - MISSION HOSPITAL,mn Mbuma Mission Hospital,MjYcpFjUjbG,dhwL8YFJUKy,CXCA_SCRN_N_TA_Age_Sex_HIV_Scrn_Visit,"CXCA_SCRN (N, TA, Age/Sex/HIVStatus/ScreenResu...",pdCeAB4EYYM,Cervical Cancer - Negative,20-24,Female,First Time Screened,0,"20-24, Female",,"20-24, Female, First Time Screened","20-24, Female, Cervical Cancer - Negative"


## Data Dictionary 
### Mapping COC Code from EHR-DATIM to DHIS2 : VlooK 
#### update 02DHIS2 COC Name,	02DHIS2 COC ID,	02EHRDATIM COC Name	,02EHRDATIM COC ID


In [88]:
# Ensure your DataFrame 'df2' and 'facility_mapping_df' are loaded as before

# Creating the mapping dictionary from 'facility_mapping_df'
data_element_to_id = dict(zip(facility_mapping_df['02EHRDATIM COC ID'], facility_mapping_df['02DHIS2 COC ID']))

# Iterating over df2 to update 'dataElementID DHIS2' based on 'DataElement Name in MRF_EHR' mapping
for index, row in df2.iterrows():
    # Use the mapping dictionary to find the corresponding ID, if available
    data_element_name = row['Disaggregation']
    if data_element_name in data_element_to_id:
        df2.at[index, 'categoryOptionComboDHIS2'] = data_element_to_id[data_element_name]
    # If the data element name is not in the dictionary, leave the value in 'dataElementID DHIS2' unchanged

# This approach directly assigns the value to 'dataElementID DHIS2', avoiding the map and potential indexing issues

### Columns Renaming 

In [90]:
df2.rename(columns={'Org Unit ID': 'Org Unit ID DHIS2'}, inplace=True)

In [91]:
df2.head(2)

,Period,SiteDistrict,orgUnit,Org Unit Name,Org Unit ID DHIS2,orgUnitDATIM,dataElementEHR,DataElement Name in MRF_EHR,Disaggregation,ElementResultGroup,ElementAge,ElementGender,ElementModality,Value,categoryOptionComboEHR,categoryOptionComboDHIS2,categoryOptionComboEHRmmd,categoryOptionComboEHRtxnewCD4
0,2025Q1,Nkayi,MBUMA - 100998 - MISSION HOSPITAL,mn Mbuma Mission Hospital,MjYcpFjUjbG,dhwL8YFJUKy,CXCA_SCRN_N_TA_Age_Sex_HIV_Scrn_Visit,"CXCA_SCRN (N, TA, Age/Sex/HIVStatus/ScreenResu...",dh4TQ68p2SC,Cervical Cancer - Negative,15-19,Female,First Time Screened,0,"15-19, Female",TLP76YUMIjH,"15-19, Female, First Time Screened","15-19, Female, Cervical Cancer - Negative"
1,2025Q1,Nkayi,MBUMA - 100998 - MISSION HOSPITAL,mn Mbuma Mission Hospital,MjYcpFjUjbG,dhwL8YFJUKy,CXCA_SCRN_N_TA_Age_Sex_HIV_Scrn_Visit,"CXCA_SCRN (N, TA, Age/Sex/HIVStatus/ScreenResu...",pdCeAB4EYYM,Cervical Cancer - Negative,20-24,Female,First Time Screened,0,"20-24, Female",Q2N7fopZXf6,"20-24, Female, First Time Screened","20-24, Female, Cervical Cancer - Negative"


## Add columns 

In [92]:
# Step 2: Find the position of the column 'orgUnit'
org_unit_index = df2.columns.get_loc('DataElement Name in MRF_EHR') + 1  # +1 to insert after 'orgUnit'

# Step 3: Insert the new columns at the desired position
df2.insert(org_unit_index, 'dataElementID DHIS2', '')  # Insert empty column for 'Org Unit Name'
#df2.insert(org_unit_index + 1, 'Org Unit ID', '')  # Insert empty column for 'Org Unit ID'

In [93]:
df2.head(2)

,Period,SiteDistrict,orgUnit,Org Unit Name,Org Unit ID DHIS2,orgUnitDATIM,dataElementEHR,DataElement Name in MRF_EHR,dataElementID DHIS2,Disaggregation,ElementResultGroup,ElementAge,ElementGender,ElementModality,Value,categoryOptionComboEHR,categoryOptionComboDHIS2,categoryOptionComboEHRmmd,categoryOptionComboEHRtxnewCD4
0,2025Q1,Nkayi,MBUMA - 100998 - MISSION HOSPITAL,mn Mbuma Mission Hospital,MjYcpFjUjbG,dhwL8YFJUKy,CXCA_SCRN_N_TA_Age_Sex_HIV_Scrn_Visit,"CXCA_SCRN (N, TA, Age/Sex/HIVStatus/ScreenResu...",,dh4TQ68p2SC,Cervical Cancer - Negative,15-19,Female,First Time Screened,0,"15-19, Female",TLP76YUMIjH,"15-19, Female, First Time Screened","15-19, Female, Cervical Cancer - Negative"
1,2025Q1,Nkayi,MBUMA - 100998 - MISSION HOSPITAL,mn Mbuma Mission Hospital,MjYcpFjUjbG,dhwL8YFJUKy,CXCA_SCRN_N_TA_Age_Sex_HIV_Scrn_Visit,"CXCA_SCRN (N, TA, Age/Sex/HIVStatus/ScreenResu...",,pdCeAB4EYYM,Cervical Cancer - Negative,20-24,Female,First Time Screened,0,"20-24, Female",Q2N7fopZXf6,"20-24, Female, First Time Screened","20-24, Female, Cervical Cancer - Negative"


In [94]:
df3 = df2 

## Data Dictionary 
### Mapping DataElement Name in MRF_EHR to dataElementID DHIS2: Data Element ID
#### fully already updated

In [96]:
# Ensure your DataFrame 'df2' and 'facility_mapping_df' are loaded as before

# Creating the mapping dictionary from 'facility_mapping_df'
data_element_to_id = dict(zip(facility_mapping_df['01DataElementNameDict'], facility_mapping_df['01DataElementsID_Dict']))

# Iterating over df2 to update 'dataElementID DHIS2' based on 'DataElement Name in MRF_EHR' mapping
for index, row in df3.iterrows():
    # Use the mapping dictionary to find the corresponding ID, if available
    data_element_name = row['DataElement Name in MRF_EHR']
    if data_element_name in data_element_to_id:
        df3.at[index, 'dataElementID DHIS2'] = data_element_to_id[data_element_name]
    # If the data element name is not in the dictionary, leave the value in 'dataElementID DHIS2' unchanged

# This approach directly assigns the value to 'dataElementID DHIS2', avoiding the map and potential indexing issues


### NEGATIVE VALUES 

In [98]:
# Filter the DataFrame for rows where 'Value' is negative
negative_values_df = df3[df3['Value'] < 0]

# Save the filtered DataFrame to a CSV file
negative_values_df.to_csv('Q4negative_values3.csv', index=False)

## DELETE NEGATIVE VALUES 

In [100]:
# Remove rows where 'Value' is negative
df3 = df3[df3['Value'] >= 0]

In [101]:
df3.head(2)

,Period,SiteDistrict,orgUnit,Org Unit Name,Org Unit ID DHIS2,orgUnitDATIM,dataElementEHR,DataElement Name in MRF_EHR,dataElementID DHIS2,Disaggregation,ElementResultGroup,ElementAge,ElementGender,ElementModality,Value,categoryOptionComboEHR,categoryOptionComboDHIS2,categoryOptionComboEHRmmd,categoryOptionComboEHRtxnewCD4
0,2025Q1,Nkayi,MBUMA - 100998 - MISSION HOSPITAL,mn Mbuma Mission Hospital,MjYcpFjUjbG,dhwL8YFJUKy,CXCA_SCRN_N_TA_Age_Sex_HIV_Scrn_Visit,"CXCA_SCRN (N, TA, Age/Sex/HIVStatus/ScreenResu...",XWK6yAwhol8,dh4TQ68p2SC,Cervical Cancer - Negative,15-19,Female,First Time Screened,0,"15-19, Female",TLP76YUMIjH,"15-19, Female, First Time Screened","15-19, Female, Cervical Cancer - Negative"
1,2025Q1,Nkayi,MBUMA - 100998 - MISSION HOSPITAL,mn Mbuma Mission Hospital,MjYcpFjUjbG,dhwL8YFJUKy,CXCA_SCRN_N_TA_Age_Sex_HIV_Scrn_Visit,"CXCA_SCRN (N, TA, Age/Sex/HIVStatus/ScreenResu...",XWK6yAwhol8,pdCeAB4EYYM,Cervical Cancer - Negative,20-24,Female,First Time Screened,0,"20-24, Female",Q2N7fopZXf6,"20-24, Female, First Time Screened","20-24, Female, Cervical Cancer - Negative"


In [102]:
#df3.to_csv('Q1 MER dataOutput before to DHIS2_P3.csv', sep=',',index=False)

## Remove uncessary columns : CONVERT TO DATIM FORMAT 

In [103]:
# List the columns you want to drop
columns_to_drop = ['SiteDistrict', 'orgUnit', 'orgUnit','Org Unit Name','orgUnitDATIM',
                   'dataElementEHR','DataElement Name in MRF_EHR','Disaggregation',
                   'ElementResultGroup','ElementAge','ElementGender','ElementModality',
                   'ElementAge','ElementModality','categoryOptionComboEHR','categoryOptionComboEHRmmd'
                   ,'categoryOptionComboEHRmmd','categoryOptionComboEHRtxnewCD4'] 

# Drop the specified columns
df3 = df3.drop(columns=columns_to_drop)

In [104]:
df3.head(3)

,Period,Org Unit ID DHIS2,dataElementID DHIS2,Value,categoryOptionComboDHIS2
0,2025Q1,MjYcpFjUjbG,XWK6yAwhol8,0,TLP76YUMIjH
1,2025Q1,MjYcpFjUjbG,XWK6yAwhol8,0,Q2N7fopZXf6
2,2025Q1,MjYcpFjUjbG,XWK6yAwhol8,0,zFPtmCibOXk


In [105]:
df3.rename(columns={'dataElementID DHIS2': 'DataElement'}, inplace=True)
df3.rename(columns={'Period': 'Period'}, inplace=True)
df3.rename(columns={'Org Unit ID DHIS2': 'OrgUnit'}, inplace=True)
df3.rename(columns={'categoryOptionComboDHIS2': 'CategoryOptionCombo'}, inplace=True)
df3.rename(columns={'value': 'Value'}, inplace=True)

In [106]:
df3.head(3)

,Period,OrgUnit,DataElement,Value,CategoryOptionCombo
0,2025Q1,MjYcpFjUjbG,XWK6yAwhol8,0,TLP76YUMIjH
1,2025Q1,MjYcpFjUjbG,XWK6yAwhol8,0,Q2N7fopZXf6
2,2025Q1,MjYcpFjUjbG,XWK6yAwhol8,0,zFPtmCibOXk


In [107]:
#df3.to_csv('DATIM FORMAT.csv', index=False)

### Drop rows where 'OrgUnit' is empty

In [108]:
# Replace empty strings or purely whitespace strings with NaN, then drop rows with NaN in the specific column
df3['OrgUnit'] = df3['OrgUnit'].replace(r'^\s*$', pd.NA, regex=True)
df3.dropna(subset=['OrgUnit'], inplace=True)

### add column

In [109]:
# Step 2: Find the position of the column 'orgUnit'
org_unit_index = df3.columns.get_loc('CategoryOptionCombo') + 1  # +1 to insert after 'orgUnit'

# Step 3: Insert the new columns at the desired position
df3.insert(org_unit_index, 'AttributeOptionCombo', '')  # Insert empty column for 'Org Unit Name'

## UPDATE THE REIOD 

In [110]:
df3.head(4)

,Period,OrgUnit,DataElement,Value,CategoryOptionCombo,AttributeOptionCombo
0,2025Q1,MjYcpFjUjbG,XWK6yAwhol8,0,TLP76YUMIjH,
1,2025Q1,MjYcpFjUjbG,XWK6yAwhol8,0,Q2N7fopZXf6,
2,2025Q1,MjYcpFjUjbG,XWK6yAwhol8,0,zFPtmCibOXk,
3,2025Q1,MjYcpFjUjbG,XWK6yAwhol8,0,n5abDKM4noW,


In [115]:
# Save the updated DataFrame back to a CSV file if needed
df3.to_csv('2025Q1 DATIM DATA TO DHIS2 october to December 12_03_2024 FINAL.csv', index=False)